In [ ]:
# ---------------- Imports ----------------
import os
import json
from collections import Counter
from collections import defaultdict

import yaml
import pandas as pd
from transformers import AutoTokenizer
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl
import os

# Path to font
FONT_DIR = os.path.join("../../config", "fonts", "linux_libertine")
FONT_PATH = os.path.join(FONT_DIR, "LinLibertine_R.ttf")

# Register font
fm.fontManager.addfont(FONT_PATH)

# Get font name
libertine_font = fm.FontProperties(fname=FONT_PATH).get_name()

# Set globally
mpl.rcParams.update({
    "font.family": libertine_font,
    "pdf.fonttype": 42,
})


In [ ]:
# ---------------- Args ----------------
VAR_SELECTIONS = [
    "mean_supports_prob_on_refutes",
    "balanced_accuracy"
]



FRAMING_CHOICE = "authoritative"
MODEL_CHOICE = "Qwen/Qwen2.5-14B-Instruct"
ALPHA_FILES = {
    1.0: [
        "20260425t162520-20260424T2359-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-1.0",
        "20260426t135136-20260425T2323-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-1.0",
        "20260426t135433-20260426T0005-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-1.0",
    
    ],

    0.7: [
        "20260425t223217-20260425T1722-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.7",
        "20260425t223515-20260425T1803-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.7",
        "20260425t223812-20260425T1845-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.7",

    ],
    
    0.5: [
        "20260425t224110-20260425T1926-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.5",
        "20260425t224407-20260425T2008-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.5",
        "20260425t224705-20260425T2049-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.5",
 
    ],

    0.3: [
        "20260426t135731-20260426T0047-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3",
        "20260426t140029-20260426T0129-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3",
        "20260426t140327-20260426T0211-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3",

    ],

    0.1: [
        "20260425t225002-20260425T2131-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.1",
        "20260426t140625-20260426T0254-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.1",
        "20260426t140923-20260426T0336-qwen2.5-14b-instruct-20260115T095923-combined-claims-15k-authoritative-0.1",

    ],

    #0.0: [
    #
    #],
}








In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]


RESULTS_DIR = os.path.join(PROJ_STORE, "evaluation", "diagnostic-individual-tables", MODEL_CHOICE)

# OUTPUT
OUTPUT_DIR = os.path.join(PROJ_STORE, "evaluation", "alpha-plot", MODEL_CHOICE)
os.makedirs(OUTPUT_DIR, exist_ok=True)






In [ ]:
# Workspace
def extract_model_name(filename):
    parts = filename.split("-")

    if len(parts) < 6:
        raise ValueError(f"Unexpected filename format: {filename}")

    return "-".join(parts[2:5])


def load_mspr_for_alpha(file_list):


    values = []

    for fname in file_list:

        full_path = os.path.join(RESULTS_DIR, f"{fname}.csv")

        if not os.path.exists(full_path):
            raise FileNotFoundError(f"Missing: {full_path}")

        df = pd.read_csv(full_path)

        auth_row = df[df["framing_type"] == FRAMING_CHOICE]

        if len(auth_row) != 1:
            raise ValueError(f"Authoritative row missing in {full_path}")

        msp = auth_row[VAR_SELECTION].iloc[0]

        values.append(msp)

    return np.array(values)



def collect_results_multi(alpha_files, var_names):

    results = {v: {} for v in var_names}
    model_names_global = set()

    for alpha, files in alpha_files.items():

        model_names_local = set()

        for fname in files:

            # --- extract model ---
            model_name = extract_model_name(fname)
            model_names_local.add(model_name)
            model_names_global.add(model_name)

            full_path = os.path.join(RESULTS_DIR, f"{fname}.csv")

            if not os.path.exists(full_path):
                raise FileNotFoundError(f"Missing: {full_path}")

            df = pd.read_csv(full_path)

            row = df[df["framing_type"] == FRAMING_CHOICE]

            if row.empty:
                raise ValueError(f"Row missing in {full_path}")

            for var in var_names:
                if var not in df.columns:
                    raise KeyError(f"{var} missing in {full_path}")

                results[var].setdefault(alpha, []).append(row[var].iloc[0])

        # enforce same model per alpha
        if len(model_names_local) != 1:
            raise ValueError(f"Inconsistent models for alpha={alpha}: {model_names_local}")

    # enforce same model globally
    if len(model_names_global) != 1:
        raise ValueError(f"Different models across alphas: {model_names_global}")

    model_name = model_names_global.pop()

    # convert lists → numpy
    for var in results:
        for alpha in results[var]:
            results[var][alpha] = np.array(results[var][alpha])

    return results, model_name




In [ ]:

# PLOT
   
def plot_alpha_curve(results):

    alphas = sorted(next(iter(results.values())).keys(), reverse=True)

    plt.figure(figsize=(5, 5))

    for var, color, marker, label in [
        ("mean_supports_prob_on_refutes", "#004C80", "o", "MSPR"),
        ("balanced_accuracy", "#4A4A4A", "s", "BAcc"),
    ]:

        means = []
        stds = []

        for a in alphas:
            vals = results[var][a] * 100.0  # convert to %
            means.append(vals.mean())
            stds.append(vals.std(ddof=1) if len(vals) > 1 else 0.0)

        plt.errorbar(
            alphas,
            means,
            yerr=stds,
            marker=marker,
            capsize=7,
            color=color,
            label=label,
            linewidth=2
        )

    plt.gca().invert_xaxis()

    plt.xlabel("Alpha (Loss Weight)", fontsize=22)
    plt.ylabel("Percentage (%)", fontsize=22)

    #plt.title("Framing Sensitivity vs Alpha", fontsize=22)

    plt.legend(loc="upper left", fontsize=18)

    plt.xticks(fontsize=22)
    plt.yticks(fontsize=22)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_FILE}.pdf", format="pdf", bbox_inches="tight")
    plt.show()



In [ ]:
# MAIN

VARS = [
    "mean_supports_prob_on_refutes",
    "balanced_accuracy"
]

results, model_name = collect_results_multi(ALPHA_FILES, VARS)

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    f"{model_name}-{FRAMING_CHOICE}-alpha-var-plot".replace("_", "-")
)

print("Results:")
print("-" * 40)

for alpha in sorted(results[VARS[0]].keys()):

    msp = results["mean_supports_prob_on_refutes"][alpha].mean() * 100
    bal = results["balanced_accuracy"][alpha].mean() * 100

    print(
        f"alpha={alpha:>4} | "
        f"MSPR={msp:.2f}% | "
        f"BAL={bal:.2f}%"
    )

print("-" * 40)


rows = []

for alpha in sorted(results["mean_supports_prob_on_refutes"].keys()):
    row = {
        "alpha": alpha,
        "MSPR_mean": results["mean_supports_prob_on_refutes"][alpha].mean() * 100,
        "MSPR_std": results["mean_supports_prob_on_refutes"][alpha].std(ddof=1) * 100,
        "BAL_mean": results["balanced_accuracy"][alpha].mean() * 100,
        "BAL_std": results["balanced_accuracy"][alpha].std(ddof=1) * 100,
    }
    rows.append(row)

df = pd.DataFrame(rows)

df = df.sort_values(by="alpha", ascending=False)

df.to_csv(f"{OUTPUT_FILE}-data.csv", index=False)

plot_alpha_curve(results)

